Nota: tempo de execução -> alterar tipo de tempo de execução -> GPU

In [1]:
############################################################################################################################################################################
# Data Exploration & Preprocessing

# -> In-depth data exploration using Pandas (data types, missing values, distributions, basic statistics).
# -> Data cleaning and preprocessing (handling missing values, data transformations).


#----- Importing libraries ----------------------------------------------------------------------------------------------------------------------------------------------------

import numpy as np # array-processing package
import pandas as pd # data analysis toolkit
import matplotlib.pyplot as plt # static, animated and interactive visualizations
import seaborn as sns # data visualization library
import re # regular expressions
from sklearn.metrics.pairwise import cosine_similarity # computes cosine similarity between samples
from sklearn.preprocessing import StandardScaler # standardizes features by removing the mean and scaling to unit variance
from scipy.sparse import csr_matrix # sparse matrix package for numeric data
from scipy.stats import pearsonr # computes the pearson correlation coefficient
from scipy.sparse.linalg import svds # singular value decomposition for matrix factorization
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score, recall_score, f1_score # evaluation metrics
from sklearn.model_selection import train_test_split # splits arrays or matrices into random train and test subsets

In [ ]:
#----- Loading the datasets ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: the datasets were saved as CSV UTF-8 files; sep or delimiter
#imdb_movies = pd.read_csv('imdb_movies.csv', encoding = 'UTF-8', sep = ';')
#imdb_ratings = pd.read_csv('imdb_ratings.csv', encoding = 'UTF-8', sep = ';')


# Note: in google colab in order to not install the libraries every time, we can use: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/projeto/'

imdb_movies = pd.read_csv(path + 'imdb_movies.csv', encoding = 'UTF-8', sep = ';')
imdb_ratings = pd.read_csv(path + 'imdb_ratings.csv', encoding = 'UTF-8', sep = ';')


# Note: .head() doesn't print automatically in the terminal, unless we use e.g. Jupiter Notebook
#print(imdb_movies.head())
#print(imdb_ratings.head())


#imdb_movies.info()
#print('-----------------------------------')
#imdb_ratings.info()



#----- Merging the datasets ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: inner join returns matching values in both tables
imdb_merged = pd.merge(imdb_movies, imdb_ratings, how = "inner", on = "imdb_title_id")
#print(imdb_merged.head())
#print(imdb_merged.info())


#print(f"Colunas: {list(imdb_merged.columns)}")
#print(len(imdb_merged.columns))

Mounted at /content/drive


In [ ]:
#----- Missing values ----------------------------------------------------------------------------------------------------------------------------------------------------

missing = imdb_merged.isnull().sum()
#print(missing)

# or

#print('---------------------------------------')
missing_table = pd.DataFrame({'Columns': missing.index, 'Missing values': missing.values})
#print(missing_table)


#----- Dropping columns ----------------------------------------------------------------------------------------------------------------------------------------------------

size = len(imdb_merged)


# Note: inplace = True modifies the original DataFrame
for column in imdb_merged.columns:
    if imdb_merged[column].isnull().sum() > 0.5*size:
        imdb_merged.drop(columns = [column], inplace = True)


# Remove columns that are not useful
imdb_merged.drop(columns = ["production_company", "year"], inplace = True) # date_published has more information and less missing values


#print(f"Colunas restantes: {list(imdb_merged.columns)}")
#print(len(imdb_merged.columns))
#print(imdb_merged.info())

In [ ]:
#----- Handle missing values ----------------------------------------------------------------------------------------------------------------------------------------------------

# Filter columns that still have missing values after dropping some columns
remaining_missing = imdb_merged.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]


remaining_missing_table = pd.DataFrame({'Columns': remaining_missing.index, 'Missing values': remaining_missing.values})
#print(remaining_missing_table)


# Numeric (float and int) columns with missing values
# Note: .iloc[0] returns the first mode in case there are more than one; fillna() replaces missing values
numeric = imdb_merged.select_dtypes(include = ['number']).columns
imdb_merged[numeric] = imdb_merged[numeric].fillna(imdb_merged[numeric].mode().iloc[0])


# Complement country and language
# Note: agg() applies a function to the group; lambda is an anonymous function; empty checks if the mode is empty
country_mode_per_language = imdb_merged.groupby("language")["country"].agg(lambda x: x.mode()[0] if not x.mode().empty else "NA")
language_mode_per_country = imdb_merged.groupby("country")["language"].agg(lambda x: x.mode()[0] if not x.mode().empty else "NA")


# Fill country with the mode of language
# Note: .map() maps values of series according to input correspondence
imdb_merged.loc[imdb_merged["country"].isna(), "country"] = imdb_merged["language"].map(country_mode_per_language)
# Fill language with the mode of country
imdb_merged.loc[imdb_merged["language"].isna(), "language"] = imdb_merged["country"].map(language_mode_per_country)
# Fill the remaining missing values with NA
imdb_merged.fillna({"country": "NA", "language": "NA"}, inplace = True)


# Handle date_published - Inês
# Check if the date_published column is in the format dd-mm-yyyy
def transform_date_format(x):

    if isinstance(x, str):

        if re.match(r'^\d{4}-\d{2}-\d{2}$', x): # if the date is in the format yyyy-mm-dd
            date = pd.to_datetime(x)
            return date.strftime('%d-%m-%Y')

        elif re.match(r'^\d{4}$', x):  # if the date is only the year
            return f"01-01-{x}"  # return a date with day 01 and month 01: here it can become random if we want, but it is not relevant

    return x # if the date is already in the format dd-mm-yyyy


imdb_merged['date_published'] = imdb_merged['date_published'].apply(transform_date_format)

#print(imdb_merged.info())

In [ ]:
#----- Handle data types - Miguel ----------------------------------------------------------------------------------------------------------------------------------------------------

# Note: optimization of the data types reduces memory usage and improves processing speed while preserving the necessary precision

#for coluna in imdb_merged.select_dtypes(include = ["int64"]).columns:
#    print(f"{coluna}: min = {imdb_merged[coluna].min()}, max = {imdb_merged[coluna].max()}")

#for coluna in imdb_merged.select_dtypes(include = ["float64"]).columns:
#    print(f"{coluna}: min = {imdb_merged[coluna].min()}, max = {imdb_merged[coluna].max()}")

#print(np.iinfo(np.int8))
#print(np.iinfo(np.int16))
#print(np.iinfo(np.int32))

# Floats
for coluna in imdb_merged.select_dtypes(include = ["float64"]).columns:
    imdb_merged[coluna] = imdb_merged[coluna].astype("float32")

    if re.search(r"avg|rating|metascore|mean", coluna):
        imdb_merged[coluna] = imdb_merged[coluna].astype("float16")

    else:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int64")

# Integers
for coluna in imdb_merged.select_dtypes(include = ["int64"]).columns:
    minimo = imdb_merged[coluna].min()
    maximo = imdb_merged[coluna].max()

    if minimo >= -128 and maximo <= 127:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int8")

    elif minimo >= -32768 and maximo <= 32767:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int16")

    elif minimo >= -2147483648 and maximo <= 2147483647:
        imdb_merged[coluna] = imdb_merged[coluna].astype("int32")

#print(imdb_merged.info()) # check the size of the dataset and the changed types


In [ ]:
############################################################################################################################################################################
# Algorithm Selection & Literature Review
# Basic Implementation & Evaluation Setup

# Note: against what was expected, the colum 'original_title' has more titles in english than 'title'


#----- User-Based Collaborative Filtering ----------------------------------------------------------------------------------------------------------------------------------------------------

# User based filtering- recommend products to a user that similar users have liked.
# For measuring the similarity between two users we can either use pearson correlation or cosine similarity.


# Select relevant columns
movies = imdb_merged[['imdb_title_id', 'original_title', 'avg_vote', 'total_votes']]


# Number of dummy users
# Note: use a small number, otherwise it will take too long to run the code
num_users = 30
user_ids = [f'user_{i}' for i in range(1, num_users + 1)]


# Generate user ratings based on avg_vote and total_votes
# Note: the ratings are generated randomly, but the probability of rating a movie is proportional to the total number of votes
ratings = []

# Note: np.random.rand() generates random numbers between 0 and 1
# Note: np.random.normal() draws random samples from a normal distribution
# Note: np.clip() limits the values in an array
for user in user_ids:
    for _, row in movies.iterrows():
        if np.random.rand() < min(1, row['total_votes'] / 100000): # normalized probability
            # Assign a rating close to the avg_vote with small variation
            rating = np.clip(np.random.normal(row['avg_vote'], 1), 1, 10) # normal distribution around avg_vote
            ratings.append([user, row['imdb_title_id'], round(rating)])


# Dataframe with the dummy ratings
ratings_df = pd.DataFrame(ratings, columns = ['user_id', 'imdb_title_id', 'rating'])
ratings_df.to_csv("dummy_ratings.csv", index = False)
print("Dummy ratings saved to 'dummy_ratings.csv'")


# Pivot the ratings dataframe to create a user-item matrix
ratings_df = pd.read_csv("dummy_ratings.csv")
user_movie_matrix = ratings_df.pivot_table(index = "user_id", columns = "imdb_title_id", values = "rating")
user_movie_matrix = user_movie_matrix.fillna(0) # no rating given = 0


# Choose the similarity method
similarity_method = 'cosine' # or 'pearson'

if similarity_method == 'cosine':
    # Note: cosine_similarity() returns a matrix of shape (n_samples, n_samples) containing the pairwise cosine similarity scores
    user_similarity = cosine_similarity(user_movie_matrix)
    user_sim_df = pd.DataFrame(user_similarity, index = user_movie_matrix.index, columns = user_movie_matrix.index)

elif similarity_method == 'pearson':
    # Note: the result is a square matrix with the same number of rows and columns as the number of users
    user_similarity = user_movie_matrix.T.corr(method = 'pearson') # transpose the matrix to get user-user similarity and not item-item similarity
    user_sim_df = pd.DataFrame(user_similarity, index = user_movie_matrix.index, columns = user_movie_matrix.index)


# Recommend movies based on user similarity
# Note: loc accesses a group of rows and columns by labels
# Note: isin() returns a boolean array, which is used to filter the dataframe
def recommend_movies(user_id, n = 5):
    if user_id not in user_sim_df.index:
        return f"User {user_id} not found."

    # Show the movies rated by the user to understand if the recommendations make sense
    user_rated_movies = user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0]
    rated_movies_info = imdb_merged[imdb_merged['imdb_title_id'].isin(user_rated_movies.index)][['original_title', 'description', 'avg_vote']]
    rated_movies_info = rated_movies_info.sort_values(by = 'avg_vote', ascending = False)

    print(f"Movies rated by {user_id}:\n")
    print(rated_movies_info)
    print("\n------------------------------\n")

    # Find similar users, excluding the user itself
    similar_users = user_sim_df[user_id].drop(user_id).sort_values(ascending = False)
    top_user = similar_users.index[0] # most similar user - top similar user

    # Recommend movies rated by top_user that the target user hasn't rated
    user_movies = set(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)
    top_user_movies = set(user_movie_matrix.loc[top_user][user_movie_matrix.loc[top_user] > 0].index)
    recommendations = list(top_user_movies - user_movies)

    # Return top n recommendations
    return imdb_merged[imdb_merged['imdb_title_id'].isin(recommendations)][['original_title', 'description', 'avg_vote']].head(n)


# Tryout
print(recommend_movies('user_1'))

Dummy ratings saved to 'dummy_ratings.csv'
Movies rated by user_1:

                 original_title  \
52108          Jibon Theke Neya   
28453  The Shawshank Redemption   
15528             The Godfather   
48078           The Dark Knight   
16556    The Godfather: Part II   
...                         ...   
12724  Manos: The Hands of Fate   
55494            Disaster Movie   
82547                    Race 3   
84505            Alien Predator   
73414          Saving Christmas   

                                             description  avg_vote  
52108  A political satire of Bangladesh under the rul...  9.398438  
28453  Two imprisoned men bond over a number of years...  9.296875  
15528  The aging patriarch of an organized crime dyna...  9.203125  
48078  When the menace known as the Joker wreaks havo...  9.000000  
16556  The early life and career of Vito Corleone in ...  9.000000  
...                                                  ...       ...  
12724  A family gets lost on

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# ----- Item-Based Collaborative Filtering ----------------------------------------------------------------------------------------------------------------------------------------------------

# Item Based Collaborative Filtering - recommends items based on similarity with the items that the target user rated.
# The similarity can be computed with Pearson Correlation or Cosine Similarity.


# Choose the similarity method
method = 'pearson' # or 'cosine'


# Select relevant columns of a random sample
# Note: we can't use the entire dataset, otherwise it will take too long to run the code
imdb_merged = imdb_merged.sample(n = 10000, random_state = 3112) # set seed
ratings = imdb_merged[['original_title', 'avg_vote']]


# Pivot table: each movie as a row, avg_vote as features
movie_ratings = ratings.pivot_table(index = 'original_title', values = 'avg_vote')


# Normalize the ratings
# Note: StandardScaler standardizes features by removing the mean and scaling to unit variance
# Note: fit_transform() fits to data, then transforms it
scaler = StandardScaler()
movie_ratings_scaled = scaler.fit_transform(movie_ratings)


# Convert to sparse matrix to save memory
# Note: csr_matrix() compresses the sparse matrix
movie_ratings_sparse = csr_matrix(movie_ratings_scaled)


# Compute similarity matrix
if method == 'cosine':
    # Cosine similarity
    similarity_matrix = cosine_similarity(movie_ratings_sparse)
    similarity_df = pd.DataFrame(similarity_matrix, index = movie_ratings.index, columns = movie_ratings.index)

elif method == 'pearson':
    # Pearson correlation
    df_ratings = pd.DataFrame(movie_ratings_scaled, index = movie_ratings.index)
    similarity_df = df_ratings.T.corr(method = 'pearson')


# Recommend 'n' most similar movies to a given movie title
def recommend_movies(movie_title, n = 5):
    if movie_title not in similarity_df.index:
        return f"Movie {movie_title} not found."

    similarities = similarity_df.loc[movie_title] # similarity scores for the movie
    sorted_similar = similarities.sort_values(ascending = False) # sorted by most to least similar
    top_similar = sorted_similar.drop(labels = [movie_title]).head(n).index.tolist() # exclude the movie itself

    recommendations = imdb_merged[imdb_merged['original_title'].isin(top_similar)][['original_title', 'description', 'avg_vote']]
    recommendations = recommendations.set_index('original_title').loc[top_similar].reset_index()

    return recommendations


# Tryout
# Note: we can't select a movie manually that is not in the sample
movie_example = imdb_merged.sample(n = 1, random_state = 3112) # set seed
print("Selected movie:")
print(f"title: {movie_example['original_title'].values[0]}, description: {movie_example['description'].values[0]}, avg_vote: {movie_example['avg_vote'].values[0]}")
print("\n--------------------\n")

recommendations = recommend_movies(movie_example['original_title'].values[0], n = 5)
print("Movies recommendations:")
print(recommendations)


# ERRO: OS TITULOS DOS FILMES NÃO ESTÃO TODOS CORRETOS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Selected movie:
title: Nebraska, description: An aging, booze-addled father makes the trip from Montana to Nebraska with his estranged son in order to claim a million-dollar Mega Sweepstakes Marketing prize., avg_vote: 7.69921875

--------------------

Movies recommendations:
          original_title                                        description  \
0  #73, Shaanthi Nivaasa  Raghu, who enters a dysfunctional family with ...   
1                 #NAME?  The Clements father and son live by the genero...   
2                  #Roxy  When Cyrus Nollen joins forces with high schoo...   
3          'A' gai wak 2  Dragon is now transferred to be the police hea...   
4        'Chi chi' de ai  Didi is an actress trying to prove herself bef...   

   avg_vote  
0  7.101562  
1  5.601562  
2  5.000000  
3  7.101562  
4  5.101562  


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
#----- Model-Based Collaborative - Matrix Factorization ------------------------------------------------------------------------------------------------------------------------------

# Matrix Factorization - decomposes the user-item matrix into two lower-dimensional matrices, one for users and one for items.

# Ensure consistency between user-movie matrix and movie metadata
valid_ids = imdb_merged['imdb_title_id'].unique()
filtered_matrix = user_movie_matrix.loc[:, user_movie_matrix.columns.isin(valid_ids)]

# Ensure movie metadata only contains relevant movies
relevant_movies = imdb_merged[imdb_merged['imdb_title_id'].isin(filtered_matrix.columns)].copy()
relevant_movies.drop_duplicates(subset='imdb_title_id', inplace=True)
relevant_movies.set_index('imdb_title_id', inplace=True)

# Note: k controls the dimensionality of the latent space and the level of compression
k = min(50, min(filtered_matrix.shape)-1) # number of latent factors

# SVD
# Note: U is the user matrix, S is the singular values (importance of each factor), Vt is the item matrix
def perform_svd(user_movie_matrix, k):
    U, S, Vt = svds(user_movie_matrix, k = k)
    S = np.diag(S) # convert singular values into a diagonal matrix to allow matrix multiplication
    return U, S, Vt

U, S, Vt = perform_svd(filtered_matrix.values, k)

# Predicted ratings matrix
predicted_ratings = np.dot(np.dot(U, S), Vt)
predicted_ratings_df = pd.DataFrame(predicted_ratings, index = filtered_matrix.index, columns = filtered_matrix.columns)


def recommend_movies_svd(user_id, n = 5):
    if user_id not in predicted_ratings_df.index:
        return f"User {user_id} not found."

    # Get the movies with highest predicted ratings for the user
    user_ratings = predicted_ratings_df.loc[user_id].sort_values(ascending = False)

    # Filter out movies already rated by the user
    already_rated = filtered_matrix.loc[user_id][filtered_matrix.loc[user_id] > 0].index
    recommended_movies = user_ratings.drop(index = already_rated, errors = 'ignore').head(n)

    # Filter only valid movies (that exist in metadata)
    valid_recs = recommended_movies.index.intersection(relevant_movies.index)

    recommended_df = relevant_movies.loc[valid_recs][['original_title', 'description', 'avg_vote']]
    recommended_df = recommended_df.sort_values(by = 'avg_vote', ascending = False)

    return recommended_df


# Tryout
user_example = filtered_matrix.sample(n = 1, random_state = 3112).index[0] # random user

# Movies previously rated by the user
rated_ids = filtered_matrix.loc[user_example][filtered_matrix.loc[user_example] > 0].index
rated_ids = rated_ids.intersection(relevant_movies.index)

user_movies = relevant_movies.loc[rated_ids][['original_title', 'description', 'avg_vote']]

print(f"Recommendation for: {user_example}\n")
print("Movies previously rated by the user:")
print(user_movies.sort_values(by = 'avg_vote', ascending = False)) # from most liked to least liked
print("\n--------------------\n")
print("Recommended movies:")
print(recommend_movies_svd(user_example, n = 5))


Recommendation for: user_11

Movies previously rated by the user:
                                original_title  \
imdb_title_id                                    
tt0111161             The Shawshank Redemption   
tt0073486      One Flew Over the Cuckoo's Nest   
tt0133093                           The Matrix   
tt0056058                              Seppuku   
tt0034583                           Casablanca   
...                                        ...   
tt2948566                                Bleed   
tt2467046                          Left Behind   
tt0116756                               Kazaam   
tt0116557                          Honfoglalás   
tt5988370                                 Reis   

                                                     description  avg_vote  
imdb_title_id                                                               
tt0111161      Two imprisoned men bond over a number of years...  9.296875  
tt0073486      A criminal pleads insanity and is adm

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


In [ ]:
# Evaluation Metrics

# Note: RMSE (Root Mean Square Error) and MAE (Mean Absolute Error) are common metrics for evaluating recommender systems;
# precision, recall, and F1-score are common metrics for evaluating classification models;
# RMSE and MAE are used for rating prediction; precision, recall, and F1-score are used for top-N recommendations


# Evaluation Functions (actual ratings vs predicted ratings)
def evaluate_rating_predictions(actual_matrix, predicted_matrix):
    mask = actual_matrix > 0
    actual = actual_matrix[mask].values.flatten()
    predicted = predicted_matrix[mask].values.flatten()

    # Remove entries where predicted or actual are nan or inf
    valid = (~np.isnan(actual)) & (~np.isnan(predicted)) & (~np.isinf(actual)) & (~np.isinf(predicted))

    actual = actual[valid]
    predicted = predicted[valid]

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return round(rmse, 4), round(mae, 4)


def evaluate_top_n(user_id, recommend_func, user_movie_matrix, imdb_merged, n=5, threshold=6.5):
    if user_id not in user_movie_matrix.index:
        return {"Precision@N": 0, "Recall@N": 0, "F1-score": 0}

    # Items rated above the threshold (relevant items)
    relevant_items = set(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] >= threshold].index)
    recommended_items = set(recommend_func(user_id, n=n)['imdb_title_id'])

    # Debug: print relevant and recommended items
    print(f"User ID: {user_id}")
    print(f"Relevant items (ratings >= {threshold}): {relevant_items}")
    print(f"Recommended items: {recommended_items}")

    if not recommended_items:
        return {"Precision@N": 0, "Recall@N": 0, "F1-score": 0}

    true_positives = len(recommended_items & relevant_items)
    precision = true_positives / len(recommended_items) if len(recommended_items) > 0 else 0  # Handle empty recommended_items
    recall = true_positives / len(relevant_items) if relevant_items else 0  # Handle empty relevant_items
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {"Precision@N": round(precision, 4), "Recall@N": round(recall, 4), "F1-score": round(f1, 4)}


def generate_model(method, user_movie_matrix, imdb_merged, similarity='cosine'):
    # USER-BASED COLLABORATIVE FILTERING
    if method == "user_based":
        if similarity == 'cosine':
            sim = cosine_similarity(user_movie_matrix)
        elif similarity == 'pearson':
            sim = user_movie_matrix.T.corr(method='pearson').fillna(0).values

        sim_df = pd.DataFrame(sim, index=user_movie_matrix.index, columns=user_movie_matrix.index)

        def recommend(user_id, n=5):
            if user_id not in sim_df.index:
                return pd.DataFrame()

            similar_users = sim_df.loc[user_id].drop(user_id).sort_values(ascending=False)
            top_user = similar_users.idxmax()
            user_seen = set(user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index)
            top_user_seen = set(user_movie_matrix.loc[top_user][user_movie_matrix.loc[top_user] > 0].index)
            recommendations = list(top_user_seen - user_seen)

            return imdb_merged[imdb_merged['imdb_title_id'].isin(recommendations)][['imdb_title_id', 'original_title']].head(n)

        # Predict ratings using similarity matrix
        weights = sim_df.div(sim_df.sum(axis=1), axis=0).fillna(0)
        predicted = weights.dot(user_movie_matrix)
        return predicted, recommend


    # ITEM-BASED COLLABORATIVE FILTERING
    elif method == "item_based":
        ratings = imdb_merged[['original_title', 'avg_vote']]
        movie_ratings = ratings.pivot_table(index='original_title', values='avg_vote')
        scaler = StandardScaler()
        movie_ratings_scaled = scaler.fit_transform(movie_ratings)
        movie_ratings_sparse = csr_matrix(movie_ratings_scaled)

        if similarity == 'cosine':
            sim_matrix = cosine_similarity(movie_ratings_sparse)
        elif similarity == 'pearson':
            df_scaled = pd.DataFrame(movie_ratings_scaled, index=movie_ratings.index)
            sim_matrix = df_scaled.T.corr(method='pearson').fillna(0).values

        sim_df = pd.DataFrame(sim_matrix, index=movie_ratings.index, columns=movie_ratings.index)

        def recommend(movie_title, n=5):
            if movie_title not in sim_df.index:
                return pd.DataFrame()

            similarities = sim_df.loc[movie_title]
            top_similar = similarities.drop(movie_title).sort_values(ascending=False).head(n).index.tolist()
            recommendations = imdb_merged[imdb_merged['original_title'].isin(top_similar)][['original_title', 'description', 'avg_vote']]
            recommendations = recommendations.set_index('original_title').loc[top_similar].reset_index()

            return recommendations

        return sim_df, recommend


    # MATRIX FACTORIZATION USING SVD
    elif method == "matrix_factorization":
        k = min(50, min(user_movie_matrix.shape)-1)
        U, S, Vt = svds(user_movie_matrix.values, k=k)
        S_diag = np.diag(S)
        predicted = np.dot(np.dot(U, S_diag), Vt)
        predicted_df = pd.DataFrame(predicted, index=user_movie_matrix.index, columns=user_movie_matrix.columns)

        def recommend(user_id, n=5):
            if user_id not in predicted_df.index:
                return pd.DataFrame()

            user_ratings = predicted_df.loc[user_id].sort_values(ascending=False)
            already_rated = user_movie_matrix.loc[user_id][user_movie_matrix.loc[user_id] > 0].index
            to_recommend = user_ratings.drop(index=already_rated).head(n).index

            return imdb_merged[imdb_merged['imdb_title_id'].isin(to_recommend)][['imdb_title_id', 'original_title']]

        return predicted_df, recommend


# Models to compare
models_to_compare = [("user_based", "cosine"), ("user_based", "pearson"), ("item_based", "cosine"), ("item_based", "pearson"), ("matrix_factorization", None)]


# Sample one user for evaluation
example_user = user_movie_matrix.sample(n=1, random_state=3112).index[0]


# Evaluation results
results = []

for method, sim in models_to_compare:
    print(f"\nEvaluating method: {method} | Similarity: {sim}")

    model_output, recommender = generate_model(method, user_movie_matrix, imdb_merged, sim) if sim else generate_model(method, user_movie_matrix, imdb_merged)
    model_output = model_output.fillna(0)

    if method == "item_based":
        # Evaluate item-based using movie-to-movie recommendations, skipping RMSE/MAE
        movie_title = imdb_merged['original_title'].sample(n=1, random_state=3112).values[0]
        print(f"Recommendations for movie: {movie_title}")
        print(recommender(movie_title, n=5))

    else:
        rmse, mae = evaluate_rating_predictions(user_movie_matrix, model_output)
        topn_metrics = evaluate_top_n(example_user, recommender, user_movie_matrix, imdb_merged, n=5)
        print(f"RMSE: {rmse}, MAE: {mae}")
        print(f"Top-N Metrics: {topn_metrics}")
        results.append({"method": method, "similarity": sim, "RMSE": rmse, "MAE": mae, **topn_metrics})


# Summary table
print("\n\nComparison Summary:")
pd.set_option("display.max_columns", None)
summary_df = pd.DataFrame(results)
print(summary_df)


# ERRO: VARIAS CORRECOES A SER FEITAS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Evaluating method: user_based | Similarity: cosine
User ID: user_11
Relevant items (ratings >= 6.5): {'tt3863552', 'tt0088763', 'tt0093010', 'tt0114558', 'tt0074253', 'tt0453556', 'tt0385002', 'tt0443680', 'tt2082197', 'tt0211915', 'tt0241303', 'tt0316356', 'tt0803061', 'tt0097523', 'tt0158714', 'tt1895587', 'tt0425210', 'tt0093771', 'tt3646462', 'tt0383694', 'tt0029870', 'tt0887912', 'tt0369702', 'tt0292963', 'tt10228168', 'tt2345737', 'tt6857112', 'tt0046438', 'tt0086197', 'tt0359950', 'tt4547056', 'tt1637688', 'tt0122933', 'tt0465494', 'tt1408101', 'tt0108002', 'tt2084970', 'tt0073195', 'tt0122151', 'tt0077416', 'tt0103776', 'tt0090967', 'tt0114388', 'tt0190332', 'tt4550098', 'tt0062512', 'tt0283139', 'tt1490017', 'tt0320661', 'tt0117802', 'tt0377109', 'tt0052311', 'tt0103874', 'tt0410764', 'tt0375679', 'tt8590896', 'tt0427470', 'tt0431308', 'tt0031505', 'tt1336617', 'tt0896872', 'tt0042332', 'tt7550000', 'tt1189073', 'tt0113277', 'tt0111161', 'tt0057129', 'tt0468492', 'tt0119670',

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


Recommendations for movie: Nebraska
          original_title                                        description  \
0               Útlaginn  From an authentic Viking saga, Outlaw tells th...   
1  #73, Shaanthi Nivaasa  Raghu, who enters a dysfunctional family with ...   
2                 #NAME?  The Clements father and son live by the genero...   
3                  #Roxy  When Cyrus Nollen joins forces with high schoo...   
4          'A' gai wak 2  Dragon is now transferred to be the police hea...   

   avg_vote  
0  6.500000  
1  7.101562  
2  5.601562  
3  5.000000  
4  7.101562  

Evaluating method: matrix_factorization | Similarity: None


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


User ID: user_11
Relevant items (ratings >= 6.5): {'tt3863552', 'tt0088763', 'tt0093010', 'tt0114558', 'tt0074253', 'tt0453556', 'tt0385002', 'tt0443680', 'tt2082197', 'tt0211915', 'tt0241303', 'tt0316356', 'tt0803061', 'tt0097523', 'tt0158714', 'tt1895587', 'tt0425210', 'tt0093771', 'tt3646462', 'tt0383694', 'tt0029870', 'tt0887912', 'tt0369702', 'tt0292963', 'tt10228168', 'tt2345737', 'tt6857112', 'tt0046438', 'tt0086197', 'tt0359950', 'tt4547056', 'tt1637688', 'tt0122933', 'tt0465494', 'tt1408101', 'tt0108002', 'tt2084970', 'tt0073195', 'tt0122151', 'tt0077416', 'tt0103776', 'tt0090967', 'tt0114388', 'tt0190332', 'tt4550098', 'tt0062512', 'tt0283139', 'tt1490017', 'tt0320661', 'tt0117802', 'tt0377109', 'tt0052311', 'tt0103874', 'tt0410764', 'tt0375679', 'tt8590896', 'tt0427470', 'tt0431308', 'tt0031505', 'tt1336617', 'tt0896872', 'tt0042332', 'tt7550000', 'tt1189073', 'tt0113277', 'tt0111161', 'tt0057129', 'tt0468492', 'tt0119670', 'tt1666185', 'tt2226417', 'tt2659414', 'tt2784678',

In [ ]:
####################################################################################################################################################################################
# Cold Start Problem

# Note: The cold start problem occurs when a recommender system cannot make accurate recommendations due to the lack of data about new users or new items.

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Cosine similarity between users
cos_sim = cosine_similarity(user_movie_matrix)

# Recommend the most popular items
def recommend_popular_items(user_movie_matrix, n_recommendations=5):
    item_popularity = user_movie_matrix.astype(bool).sum(axis=0)  # count how many users rated each item
    popular_items = item_popularity.sort_values(ascending=False).head(n_recommendations).index.tolist()  # top n most popular items
    return popular_items


# Recommend popular items for a new user
def recommend_for_new_user(user_movie_matrix, n_recommendations=5):
    return recommend_popular_items(user_movie_matrix, n_recommendations)


# Recommend similar items for an existing user based on the items they have rated
# Note: the new user falls back to most popular items
def recommend_similar_items_based_on_popularity(user_id, user_movie_matrix, n_recommendations=5):
    if user_id not in user_movie_matrix.index:
        # If user does not exist, recommend popular items
        return recommend_popular_items(user_movie_matrix, n_recommendations)

    user_ratings = user_movie_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings > 0].index  # items the user has rated
    candidate_items = user_ratings[user_ratings == 0].index  # items the user has not rated

    similar_items = []

    # For each item rated by the user, compute its similarity to unrated items
    for item in rated_items:
        item_vector = user_movie_matrix[item].values.reshape(1, -1)  # Get vector of the rated item
        candidate_vectors = user_movie_matrix[candidate_items].values.T  # Get vectors of candidate items
        similarities = cosine_similarity(item_vector, candidate_vectors).flatten()  # Calculate similarity

        # Store the candidate items and their similarity scores
        similar_items.extend(zip(candidate_items, similarities))

    # Sort items by similarity score in descending order
    similar_items = sorted(similar_items, key=lambda x: x[1], reverse=True)

    # Top n unique recommended items
    recommended = []
    seen = set()

    # Iterate over sorted similar items and add unique ones to the recommendation list
    for item, _ in similar_items:
        if item not in seen:
            recommended.append(item)
            seen.add(item)
        if len(recommended) == n_recommendations:
            break

    return recommended


# Tryout
new_user_id = 100  # This user does not exist in the matrix
popular_recommendations = recommend_for_new_user(user_movie_matrix)
similar_item_recommendations = recommend_similar_items_based_on_popularity(new_user_id, user_movie_matrix)

print(f"Popular recommendations for new user {new_user_id}: {popular_recommendations}")
print(f"Similarity-based recommendations for new user {new_user_id}: {similar_item_recommendations}")


# ERRO: VARIAS CORRECOES A SER FEITAS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Popular recommendations for new user 100: ['tt0012349', 'tt0434409', 'tt0435625', 'tt0430922', 'tt0435651']
Similarity-based recommendations for new user 100: ['tt0012349', 'tt0434409', 'tt0435625', 'tt0430922', 'tt0435651']


In [ ]:
######################################################################################################################################################
# Basic Evaluation Framework (Train/Test Split for Item-Based Collaborative Filtering)

# Note: we evaluate item-based collaborative filtering by splitting the user-item matrix (user_movie_matrix) into training and testing sets.
# Then we compute item-item similarity based on the training set, generate predictions, and evaluate them against the test set using RMSE and MAE.


# Split the ratings into training and testing sets
# Note: splitting by rows = splitting by users
user_ids = user_movie_matrix.index.tolist()
train_ids, test_ids = train_test_split(user_ids, test_size = 0.2, random_state = 3112)

train_data = user_movie_matrix.loc[train_ids]
test_data = user_movie_matrix.loc[test_ids]


# Cosine similarity (training data)
item_similarity = cosine_similarity(train_data.T) # transpose to get items as rows
item_similarity_df = pd.DataFrame(item_similarity, index = train_data.columns, columns = train_data.columns)


# Predict ratings
def predict_ratings_item_based(train_data, item_similarity):
    mean_user_rating = train_data.mean(axis = 1).values.reshape(-1, 1) # mean rating per user
    ratings_diff = train_data.sub(mean_user_rating.flatten(), axis = 0).fillna(0) # center the ratings
    pred = mean_user_rating + np.dot(ratings_diff, item_similarity) / np.abs(item_similarity).sum(axis = 1) # weighted sum of item similarities

    return pd.DataFrame(pred, index=train_data.index, columns=train_data.columns)


# Generate predictions
predicted_ratings_item = predict_ratings_item_based(train_data, item_similarity)


# Evaluate predictions (test data)
def evaluate_predictions(test_data, predicted_ratings):
    mask = test_data > 0 # only consider the test entries where ratings exist
    actual = test_data[mask].values.flatten()
    predicted = predicted_ratings.reindex(index = test_data.index, columns = test_data.columns).fillna(0)
    predicted = predicted[mask].values.flatten()

    # RMSE and MAE
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)

    print(f"Evaluation on Test Set:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")


# Summary
evaluate_predictions(test_data, predicted_ratings_item)


# ERRO: VARIAS CORRECOES A SER FEITAS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

<ipython-input-11-38e25b34969e>:26: RuntimeWarning: invalid value encountered in divide
  pred = mean_user_rating + np.dot(ratings_diff, item_similarity) / np.abs(item_similarity).sum(axis = 1) # weighted sum of item similarities


ValueError: Input contains NaN.

In [ ]:
###########################################3###############################################################################################################################
# Parameter Tuning and Parameter Settings

# User-Based Collaborative Filtering -----------------------------------------------------------------------------------------------------------------------------------

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from scipy.stats import randint
from scipy.sparse.linalg import svds


# Param grid
param_grid_user_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 20]}


# Fit the model and evaluate
def evaluate_user_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(user_movie_matrix)
    elif similarity_metric == 'pearson':
        sim_matrix = user_movie_matrix.T.corr(method = 'pearson').fillna(0).values

    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.index, columns = user_movie_matrix.index)
    weights = sim_df.div(sim_df.sum(axis = 1), axis = 0).fillna(0)
    predicted_ratings = weights.dot(user_movie_matrix)

    return predicted_ratings


# Evaluate predictions (RMSE AND MAE)
def evaluate_predictions(actual_matrix, predicted_matrix):
    mask = actual_matrix > 0
    actual = actual_matrix[mask].values.flatten()
    predicted = predicted_matrix[mask].values.flatten()

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return round(rmse, 4), round(mae, 4)


# Grid Search
grid_search_user_based = GridSearchCV(estimator = NearestNeighbors(), param_grid = param_grid_user_based, cv = 5, n_jobs = -1)
grid_search_user_based.fit(user_movie_matrix)


# Best parameters and score
print("Best parameters for User-Based with Grid Search:", grid_search_user_based.best_params_)
print("Best score:", grid_search_user_based.best_score_)



# Random Search
param_dist_user_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': randint(5, 50)}

random_search_user_based = RandomizedSearchCV(estimator = NearestNeighbors(), param_distributions = param_dist_user_based, n_iter = 100, cv = 5, n_jobs = -1)
random_search_user_based.fit(user_movie_matrix)


# Best parameters and score
print("Best parameters for User-Based with Random Search:", random_search_user_based.best_params_)
print("Best score:", random_search_user_based.best_score_)



# Item-Based Collaborative Filtering -------------------------------------------------------------------------------------------------------------------------------------------

# Param grid
param_grid_item_based = {'similarity': ['cosine', 'pearson'], 'n_neighbors': [5, 10, 20]}


# Fit the model and evaluate
def evaluate_item_based_model(user_movie_matrix, similarity_metric = 'cosine', n_neighbors = 5):
    if similarity_metric == 'cosine':
        sim_matrix = cosine_similarity(user_movie_matrix.T)
    elif similarity_metric == 'pearson':
        sim_matrix = user_movie_matrix.corr(method = 'pearson').fillna(0).values
    sim_df = pd.DataFrame(sim_matrix, index = user_movie_matrix.columns, columns = user_movie_matrix.columns)

    # Predict ratings
    weights = sim_df.div(sim_df.sum(axis = 1), axis = 0).fillna(0)
    predicted_ratings = weights.dot(user_movie_matrix.T).T
    return predicted_ratings


# Grid Search
grid_search_item_based = GridSearchCV(estimator = NearestNeighbors(), param_grid = param_grid_item_based, cv = 5, n_jobs = -1)
grid_search_item_based.fit(user_movie_matrix)

# Best parameters and score
print("Best parameters for Item-Based:", grid_search_item_based.best_params_)
print("Best score:", grid_search_item_based.best_score_)



# Matrix Factorization (SVD) ---------------------------------------------------------------------------------------------------------------------------------------

# Param grid for Matrix Factorization (SVD)
param_grid_matrix_factorization = {'k': [10, 20, 30, 50]}


# Fit the model and evaluate
def evaluate_matrix_factorization(user_movie_matrix, k = 20):
    U, S, Vt = svds(user_movie_matrix.values, k = k)
    S_diag = np.diag(S)
    predicted_ratings = np.dot(np.dot(U, S_diag), Vt)
    return pd.DataFrame(predicted_ratings, index = user_movie_matrix.index, columns = user_movie_matrix.columns)


# Grid Search
grid_search_matrix_factorization = GridSearchCV(estimator = svds, param_grid = param_grid_matrix_factorization, cv = 5, n_jobs = -1)
grid_search_matrix_factorization.fit(user_movie_matrix)


# Best parameters and score
print("Best parameters for Matrix Factorization:", grid_search_matrix_factorization.best_params_)
print("Best score:", grid_search_matrix_factorization.best_score_)

TypeError: If no scoring is specified, the estimator passed should have a 'score' method. The estimator NearestNeighbors() does not.

In [ ]:
### TAREFAS #############################################################################################################################

### fazer hybrid recommender systems
### fazer código geral


# Explore techniques to improve the scalability of the algorithm (if necessary, depending on the dataset size and implementation).
# This might involve exploring alternative data structures or optimization techniques.
# Document the implemented solutions.